# 05 - LSTM Model

Train an LSTM sequence model on `data/processed/traffic_density.csv` to predict `vehicle_count` for the next interval, using a sliding window of past intervals per camera.

Output: metrics appended to `results/model_comparison.csv`, model saved to `models/lstm.pt`

**Split:** S01, S03, S04 = train · S02 = validation · S05 = test

In [1]:
import random
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

sys.path.append("..")

from src.data_utils import (
    add_delta_target,
    delta_target_column,
    drop_incomplete_lag_rows,
    feature_columns,
    lag1_raw_column,
    load_density_data,
    scale_features,
    split_by_column,
    target_column,
)
from src.eval_metrics import log_results, print_metrics, regression_metrics
from src.sequence_utils import create_sequences

In [2]:
# Reproducibility seed
SEED = 24
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
processed_dir = Path("../data/processed")
input_path = processed_dir / "traffic_density.csv"

results_path = Path("../results/model_comparison.csv")
model_dir = Path("../models")
model_path = model_dir / "lstm.pt"

num_layers = 2
dropout = 0.2
learning_rate = 0.001
batch_size = 16
num_epochs = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
density = load_density_data(input_path)
density = drop_incomplete_lag_rows(density)
density = add_delta_target(density)

train_df, val_df, test_df = split_by_column(density)
train_df, val_df, test_df, scaler = scale_features(train_df, val_df, test_df)
print(f"train: {len(train_df)} rows, val: {len(val_df)} rows, test: {len(test_df)} rows")

train: 132 rows, val: 41 rows, test: 424 rows


In [5]:
max_train_target = train_df[target_column].max()
print(f"max_train_target: {max_train_target:.4f}")

max_train_target: 8.6000


In [6]:
class lstm_model(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            n_features, hidden_size, num_layers, batch_first=True, dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze(-1)


def make_weighted_mse_loss(max_target):
    def weighted_mse_loss(preds, targets, raw_counts):
        weights = 1.0 + 0.3 * raw_counts / max_target
        return torch.mean(weights * (preds - targets) ** 2)
    return weighted_mse_loss


loss_fn = make_weighted_mse_loss(max_train_target)

In [7]:
window_size_options = [2, 3, 5]
hidden_size_options = [16, 32]
sweep_epochs = num_epochs

In [8]:
sweep_results = []

for window_size_candidate in window_size_options:
    x_train_c, y_train_delta_c = create_sequences(train_df, window_size_candidate, target=delta_target_column)
    x_val_c, y_val_delta_c = create_sequences(val_df, window_size_candidate, target=delta_target_column)
    _, y_train_raw_c = create_sequences(train_df, window_size_candidate, target=target_column)
    _, y_val_raw_c = create_sequences(val_df, window_size_candidate, target=target_column)
    _, y_val_lag1_c = create_sequences(val_df, window_size_candidate, target=lag1_raw_column)

    train_dataset_c = TensorDataset(
        torch.tensor(x_train_c, dtype=torch.float32),
        torch.tensor(y_train_delta_c, dtype=torch.float32),
        torch.tensor(y_train_raw_c, dtype=torch.float32),
    )
    train_loader_c = DataLoader(train_dataset_c, batch_size=batch_size, shuffle=True)
    x_val_tensor_c = torch.tensor(x_val_c, dtype=torch.float32).to(device)

    for hidden_size_candidate in hidden_size_options:
        n_features = x_train_c.shape[2]
        candidate_model = lstm_model(n_features, hidden_size_candidate, num_layers, dropout).to(device)
        optimizer_c = torch.optim.Adam(candidate_model.parameters(), lr=learning_rate)

        for epoch in range(sweep_epochs):
            candidate_model.train()
            for x_batch, y_delta_batch, y_raw_batch in train_loader_c:
                x_batch = x_batch.to(device)
                y_delta_batch = y_delta_batch.to(device)
                y_raw_batch = y_raw_batch.to(device)
                optimizer_c.zero_grad()
                preds_delta = candidate_model(x_batch)
                loss = loss_fn(preds_delta, y_delta_batch, y_raw_batch)
                loss.backward()
                optimizer_c.step()

        candidate_model.eval()
        with torch.no_grad():
            val_preds_delta_c = candidate_model(x_val_tensor_c).cpu().numpy()
        val_preds_c = val_preds_delta_c + y_val_lag1_c

        val_metrics_c = regression_metrics(y_val_raw_c, val_preds_c)
        sweep_results.append(dict(window_size=window_size_candidate, hidden_size=hidden_size_candidate, n_train=len(x_train_c), n_val=len(x_val_c), **val_metrics_c))

sweep_df = pd.DataFrame(sweep_results).sort_values("rmse").reset_index(drop=True)
sweep_df

,window_size,hidden_size,n_train,n_val,mae,rmse,r2,mape
0,2,32,105,33,1.200805,1.542699,0.352686,31.262061
1,2,16,105,33,1.175185,1.574622,0.325619,30.148203
2,3,16,94,29,1.149244,1.614076,0.329255,29.960470
3,3,32,94,29,1.382468,1.905904,0.064785,35.816409
4,5,16,72,21,1.657879,2.061631,-0.105538,45.068269
5,5,32,72,21,1.676923,2.150808,-0.203247,45.257882


In [9]:
best_config = sweep_df.iloc[0]
window_size = int(best_config["window_size"])
hidden_size = int(best_config["hidden_size"])
print(f"Selected config: window_size={window_size}, hidden_size={hidden_size}")

Selected config: window_size=2, hidden_size=32


In [10]:
x_train, y_train_delta = create_sequences(train_df, window_size, target=delta_target_column)
x_val, y_val_delta = create_sequences(val_df, window_size, target=delta_target_column)
x_test, y_test_delta = create_sequences(test_df, window_size, target=delta_target_column)

_, y_train = create_sequences(train_df, window_size, target=target_column)
_, y_val = create_sequences(val_df, window_size, target=target_column)
_, y_test = create_sequences(test_df, window_size, target=target_column)

_, y_train_lag1 = create_sequences(train_df, window_size, target=lag1_raw_column)
_, y_val_lag1 = create_sequences(val_df, window_size, target=lag1_raw_column)
_, y_test_lag1 = create_sequences(test_df, window_size, target=lag1_raw_column)

print(f"train sequences: {x_train.shape}, val sequences: {x_val.shape}, test sequences: {x_test.shape}")

train sequences: (105, 2, 11), val sequences: (33, 2, 11), test sequences: (386, 2, 11)


In [11]:
train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train_delta, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

x_val_tensor = torch.tensor(x_val, dtype=torch.float32).to(device)
y_val_delta_tensor = torch.tensor(y_val_delta, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test_delta_tensor = torch.tensor(y_test_delta, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

In [12]:
n_features = x_train.shape[2]
model = lstm_model(n_features, hidden_size, num_layers, dropout).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for x_batch, y_delta_batch, y_raw_batch in train_loader:
        x_batch = x_batch.to(device)
        y_delta_batch = y_delta_batch.to(device)
        y_raw_batch = y_raw_batch.to(device)

        optimizer.zero_grad()
        preds_delta = model(x_batch)
        loss = loss_fn(preds_delta, y_delta_batch, y_raw_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * x_batch.size(0)

    epoch_loss /= len(train_dataset)
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1}/{num_epochs} - loss: {epoch_loss:.4f}")

training_time_sec = time.time() - start_time
print(f"Trained in {training_time_sec:.2f} seconds")

epoch 10/100 - loss: 1.2463
epoch 20/100 - loss: 0.8960
epoch 30/100 - loss: 0.7316
epoch 40/100 - loss: 0.6777
epoch 50/100 - loss: 0.6018
epoch 60/100 - loss: 0.6221
epoch 70/100 - loss: 0.5797
epoch 80/100 - loss: 0.5794
epoch 90/100 - loss: 0.5231
epoch 100/100 - loss: 0.4742
Trained in 1.51 seconds


In [14]:
model.eval()
with torch.no_grad():
    val_preds_delta = model(x_val_tensor).cpu().numpy()

val_preds = val_preds_delta + y_val_lag1

val_metrics = regression_metrics(y_val, val_preds)
print_metrics("lstm", "validation", val_metrics)
log_results(results_path, "lstm", "validation", val_metrics, training_time_sec)

lstm [validation] -> mae: 1.2298  rmse: 1.6186  r2: 0.2874  mape: 32.17%


In [15]:
model.eval()
with torch.no_grad():
    test_preds_delta = model(x_test_tensor).cpu().numpy()

test_preds = test_preds_delta + y_test_lag1

test_metrics = regression_metrics(y_test, test_preds)
print_metrics("lstm", "test", test_metrics)
log_results(results_path, "lstm", "test", test_metrics, training_time_sec)

lstm [test] -> mae: 1.4285  rmse: 1.9281  r2: 0.6812  mape: 92.96%


In [16]:
model_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), model_path)

print(f"Saved model to {model_path}")

Saved model to ..\models\lstm.pt


In [17]:
train_losses = []
val_losses = []

loss_model = lstm_model(n_features, hidden_size, num_layers, dropout).to(device)
loss_optimizer = torch.optim.Adam(loss_model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    loss_model.train()
    epoch_train_loss = 0.0

    for x_batch, y_delta_batch, y_raw_batch in train_loader:
        x_batch = x_batch.to(device)
        y_delta_batch = y_delta_batch.to(device)
        y_raw_batch = y_raw_batch.to(device)
        loss_optimizer.zero_grad()
        preds_delta = loss_model(x_batch)
        loss = loss_fn(preds_delta, y_delta_batch, y_raw_batch)
        loss.backward()
        loss_optimizer.step()
        epoch_train_loss += loss.item() * x_batch.size(0)

    epoch_train_loss /= len(train_dataset)

    loss_model.eval()
    with torch.no_grad():
        epoch_val_loss = loss_fn(loss_model(x_val_tensor), y_val_delta_tensor, y_val_tensor).item()

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

curves_path = Path("../results/training_curves.csv")
curves_df = pd.DataFrame({
    "model_name": "lstm",
    "epoch": range(1, num_epochs + 1),
    "train_loss": train_losses,
    "val_loss": val_losses,
})

if curves_path.exists():
    existing = pd.read_csv(curves_path)
    existing = existing[existing["model_name"] != "lstm"]
    curves_df = pd.concat([existing, curves_df], ignore_index=True)

curves_df.to_csv(curves_path, index=False)
print(f"Saved training curve to {curves_path}")

Saved training curve to ..\results\training_curves.csv


In [18]:
predictions_path = Path("../results/predictions.csv")

model.eval()
with torch.no_grad():
    test_preds_delta_final = model(x_test_tensor).cpu().numpy()

test_preds_final = test_preds_delta_final + y_test_lag1

predictions_df = pd.DataFrame({
    "model_name": "lstm",
    "split": "test",
    "actual": y_test,
    "predicted": test_preds_final,
})

if predictions_path.exists():
    existing_preds = pd.read_csv(predictions_path)
    existing_preds = existing_preds[existing_preds["model_name"] != "lstm"]
    predictions_df = pd.concat([existing_preds, predictions_df], ignore_index=True)

predictions_df.to_csv(predictions_path, index=False)
print(f"Saved test predictions to {predictions_path}")

Saved test predictions to ..\results\predictions.csv


In [19]:
loaded_model = lstm_model(n_features, hidden_size, num_layers, dropout).to(device)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()

sample_x = x_test_tensor[0:1]
sample_actual = y_test[0]
sample_lag1 = y_test_lag1[0]

with torch.no_grad():
    sample_pred_delta = loaded_model(sample_x).cpu().numpy()[0]

sample_pred = sample_pred_delta + sample_lag1

print(f"actual: {sample_actual:.4f}")
print(f"predicted: {sample_pred:.4f}")

actual: 4.8667
predicted: 4.7775
